# Notebook 01: Detection Workflow

This notebook walks through the complete weed detection pipeline provided by the
`ragweed_toolkit.detection` module. The workflow covers:

1. **Configuring a YOLO detector** with the validated hyperparameters from the Springer chapter
2. **Setting up SAHI sliced inference** for high-resolution field imagery
3. **Simulating detection results** and extracting GPS coordinates
4. **Building a GeoDataFrame** and exporting to shapefile
5. **Visualizing detections** on a map with severity symbology

All data in this notebook is synthetic -- no GPU, trained model, or field images are needed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
import os

np.random.seed(42)

## 1. Training Configuration

The `TrainingConfig` dataclass stores all hyperparameters validated in the chapter
(Section 3.2, Table 3). Defaults match the production settings so you only need
to override what you want to change.

In [ ]:
from dataclasses import asdict

# TrainingConfig lives in the detection module but only needs dataclasses
# (no GPU or ultralytics required just to inspect it)
try:
    from ragweed_toolkit.detection.trainer import TrainingConfig
except ImportError:
    # If ultralytics is not installed, define a minimal stand-in
    from dataclasses import dataclass

    @dataclass
    class TrainingConfig:
        model: str = "yolo11l.pt"
        data: str = ""
        epochs: int = 50
        imgsz: int = 640
        batch: int = 64
        device: str = "0"
        optimizer: str = "SGD"
        lr0: float = 0.01
        lrf: float = 0.0001
        momentum: float = 0.937
        warmup_epochs: int = 3
        patience: int = 15
        max_det: int = 10000
        seed: int = 42
        amp: bool = True
        box: float = 7.5
        cls: float = 0.5
        dfl: float = 1.5
        mosaic: float = 1.0
        crop_fraction: float = 0.29
        degrees: float = 15.0
        project: str = "training_results"
        name: str = ""
        workers: int = 8
        cache: bool = True
        save_period: int = 10


# Create a config with chapter defaults
cfg = TrainingConfig(
    model="yolo11l.pt",
    data="path/to/ragweed_dataset.yaml",
    device="0",
)

print("=== Default TrainingConfig (Chapter Table 3) ===")
for key, value in asdict(cfg).items():
    print(f"  {key:20s} = {value}")

In [ ]:
# Override specific hyperparameters for experimentation
cfg_custom = TrainingConfig(
    model="yolo11l.pt",
    data="path/to/ragweed_dataset.yaml",
    epochs=100,
    imgsz=1024,
    batch=32,
    device="0,1",     # dual-GPU DataParallel
    lr0=0.005,
    patience=25,
    project="training_results/custom",
    name="experiment_v2",
)

print("=== Custom TrainingConfig ===")
print(f"  Model        : {cfg_custom.model}")
print(f"  Image size   : {cfg_custom.imgsz}")
print(f"  Batch size   : {cfg_custom.batch}")
print(f"  Epochs       : {cfg_custom.epochs}")
print(f"  Device       : {cfg_custom.device}")
print(f"  LR           : {cfg_custom.lr0}")
print(f"  Patience     : {cfg_custom.patience}")

## 2. SAHI Sliced Inference Configuration

SAHI (Slicing Aided Hyper Inference) divides large images into overlapping tiles,
runs detection on each tile, and merges results with NMS. This is critical for
dense weed scenes where single-pass inference misses small objects.

Note: `max_det=10000` because YOLO's default of 300 silently drops detections
in dense agricultural fields.

In [ ]:
try:
    from ragweed_toolkit.detection.inference import SahiConfig
except ImportError:
    from dataclasses import dataclass

    @dataclass
    class SahiConfig:
        model_path: str = ""
        slice_size: int = 640
        overlap_ratio: float = 0.2
        confidence_threshold: float = 0.25
        nms_iou: float = 0.5
        max_det: int = 10000
        device: str = "cuda:0"


sahi_cfg = SahiConfig(
    model_path="weights/best.pt",
    slice_size=640,
    overlap_ratio=0.2,
    confidence_threshold=0.25,
    device="cuda:0",
)

print("=== SAHI Config (Chapter Section 3.3) ===")
print(f"  Model          : {sahi_cfg.model_path}")
print(f"  Slice size     : {sahi_cfg.slice_size}px")
print(f"  Overlap        : {sahi_cfg.overlap_ratio:.0%}")
print(f"  Confidence     : {sahi_cfg.confidence_threshold}")
print(f"  NMS IoU        : {sahi_cfg.nms_iou}")
print(f"  Max detections : {sahi_cfg.max_det}")
print(f"  Device         : {sahi_cfg.device}")

## 3. Simulate Detection Results

In production, `run_sahi()` returns a list of detection dictionaries for each
image. Here we generate synthetic detections to demonstrate the downstream
pipeline.

In [ ]:
# Simulate 200 images taken across a field
n_images = 200

# GPS coordinates in central Chile (WGS 84)
base_lat, base_lon = -34.15, -70.75
lats = base_lat + np.random.uniform(-0.005, 0.005, n_images)
lons = base_lon + np.random.uniform(-0.005, 0.005, n_images)

# Simulate ragweed (AMBEL) counts per image
# Higher density near a hotspot
dist_to_hotspot = np.sqrt((lats - (base_lat + 0.002))**2 + (lons - (base_lon - 0.001))**2)
mean_counts = 50 * np.exp(-dist_to_hotspot / 0.003) + 2
ambel_counts = np.random.poisson(mean_counts).astype(int)

# Build detection summary
detections_df = pd.DataFrame({
    "image": [f"field_{i:04d}.jpg" for i in range(n_images)],
    "latitude": lats,
    "longitude": lons,
    "nr_ambel": ambel_counts,
    "confidence_mean": np.random.uniform(0.6, 0.95, n_images),
})

print(f"Simulated {n_images} images with {ambel_counts.sum():,} total AMBEL detections")
print(f"Detections per image: min={ambel_counts.min()}, "
      f"median={int(np.median(ambel_counts))}, max={ambel_counts.max()}")
detections_df.head(10)

## 4. Build GeoDataFrame and Export to Shapefile

Convert the detection summary to a GeoDataFrame with point geometries
and export to a temporary shapefile for QGIS visualization.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# Create point geometries from GPS coordinates
geometry = [Point(lon, lat) for lon, lat in zip(detections_df["longitude"],
                                                 detections_df["latitude"])]
gdf = gpd.GeoDataFrame(detections_df, geometry=geometry, crs="EPSG:4326")

# Add severity classification (matching QGIS symbology from the chapter)
def classify_severity(count):
    if count == 0:
        return "Absent"
    elif count <= 5:
        return "Low"
    elif count <= 10:
        return "Medium"
    elif count <= 30:
        return "High"
    else:
        return "Very High"

gdf["severity"] = gdf["nr_ambel"].apply(classify_severity)

print(f"GeoDataFrame: {len(gdf)} points, CRS={gdf.crs}")
print(f"\nSeverity distribution:")
print(gdf["severity"].value_counts().to_string())

# Export to temporary shapefile
with tempfile.TemporaryDirectory() as tmpdir:
    shp_path = os.path.join(tmpdir, "detections.shp")
    gdf.to_file(shp_path)
    print(f"\nExported to: {shp_path}")
    # List generated files
    for f in sorted(os.listdir(tmpdir)):
        size = os.path.getsize(os.path.join(tmpdir, f))
        print(f"  {f:30s} {size:>8,} bytes")

## 5. Visualize Detections on a Map

Plot the detection points colored by severity level, using the ragweed toolkit's
publication-quality style settings.

In [ ]:
from ragweed_toolkit.viz.style import set_publication_style, ORARA_COLORS

set_publication_style()

# Map severity levels to the ORARA color scheme
severity_colors = {
    "Absent":    ORARA_COLORS["Ausente"],
    "Low":       ORARA_COLORS["Bajo"],
    "Medium":    ORARA_COLORS["Medio"],
    "High":      ORARA_COLORS["Alto"],
    "Very High": ORARA_COLORS["Muy Alto"],
}
severity_sizes = {
    "Absent": 10, "Low": 20, "Medium": 35, "High": 55, "Very High": 80,
}

fig, ax = plt.subplots(figsize=(10, 8))

for level in ["Absent", "Low", "Medium", "High", "Very High"]:
    mask = gdf["severity"] == level
    if mask.sum() == 0:
        continue
    subset = gdf[mask]
    ax.scatter(
        subset.geometry.x, subset.geometry.y,
        c=severity_colors[level],
        s=severity_sizes[level],
        label=f"{level} (n={mask.sum()})",
        edgecolors="black", linewidths=0.3, alpha=0.8,
    )

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Ragweed Detection Map -- Synthetic Data", fontweight="bold")
ax.legend(title="Severity", loc="upper right")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Total detections: {gdf['nr_ambel'].sum():,}")
print(f"Mean per image: {gdf['nr_ambel'].mean():.1f}")

## Summary

This notebook demonstrated the detection workflow:

1. **TrainingConfig** stores validated hyperparameters (Table 3) as a dataclass.
   To train for real, call `train(cfg)` with a GPU and dataset YAML.

2. **SahiConfig** configures sliced inference with `max_det=10000` to avoid
   silently dropping detections in dense agricultural scenes.

3. Detection results are converted to **GeoDataFrames** with GPS coordinates
   and exported as shapefiles for QGIS integration.

4. The severity color scheme (Absent/Low/Medium/High/Very High) matches the
   QGIS symbology used throughout the chapter.

### Next Steps

- **Notebook 02**: Cross-domain analysis with ResNet-50 embeddings and MMD
- **Notebook 03**: Spatial mapping with kriging and LISA clusters
- **Notebook 04**: Satellite integration with spectral indices and risk zones